# Capítulo 6. Prompt Engineering ⚙

Este capítulo explora os fundamentos do uso de modelos generativos por meio da engenharia de prompts e da verificação de saída. Ele mostra tecnicas e métodos para guiar os modelos para respostas mais precisas e otimizadas.

Nesta atividade iremos demonstrar de forma prática os principais assuntos presentes neste capítulo.


Primeiro carregamos o modelo e o tokenizador, além de prepararmos a pipeline

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
import torch

model_id = "Qwen/Qwen2.5-3B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype="auto",
    device_map="auto"
)

print("modelo e tokenizador carregados com sucesso!")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

modelo e tokenizador carregados com sucesso!


In [ ]:
# Cria a pipeline
pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    return_full_text=False,
    max_new_tokens=500,
    max_length=None,
    do_sample=False # Faz com que a saída seja consistente ao não realizar a amostragem de probabilidade e selecionando apenas o mais provavel
)

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Passing `generation_config` together with generation-related arguments=({'max_length', 'max_new_tokens', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


Exemplo de prompt para testar o modelo:

In [ ]:
# Exemplo de prompt que será usado
messages = [
{"role": "user", "content": "Crie uma piada sobre galinhas."}]

# Saída gerada a partir do prompt entregue ao modelo
output = pipe(messages)
print(output[0]["generated_text"])

Por acaso, aqui está uma piada sobre galinhas para você:

Por que as galinhas são tão boas em jogos de cartas?

Porque elas sabem que "pato" é só um "pato"!


# Temperaura

Temperatura é o parâmetro que controla a criatividade ou aleatoriedade nas respostas. Seguindo este conceito temos que quanto menor a temperatura mais provável que as respostas sejam as mesmas e vice-versa, quanto mais a temperatura maior a criatividade e aleatoriedade nas respostas.

In [ ]:
# Usando uma alta temperatura
output = pipe(messages, do_sample=True, temperature=1)
print(output[0]["generated_text"])

# Usando uma baixa temperatura
output = pipe(messages, do_sample=True, temperature=0.1)
print(output[0]["generated_text"])


Passing `generation_config` together with generation-related arguments=({'temperature', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


Aqui está uma piada sobre galinhas para você:

Por que as galinhas são ótimas em reuniões?

Porque elas sempre trazem o assunto ao ponto!
Por acaso, aqui está uma piada sobre galinhas para você:

Por que as galinhas são tão boas em jogos de cartas?

Porque elas sabem que "pato" é só um "pato"!


# Top_p

Representa o limite de probabilidade no cálculo do próximo tokem durante a geração do texto, diminuindo ou aumentando quantidade de tokens que o modelo pode considerar na sua resposta.

In [ ]:
# Usando um alto top_p
output = pipe(messages, do_sample=True, top_p=1)
print(output[0]["generated_text"])

# Usando um baixo top_p
output = pipe(messages, do_sample=True, top_p=0.1)
print(output[0]["generated_text"])

Por que as galinhas não entendem piadas? Porque elas têm muitos ovos mentais!
Por acaso, aqui está uma piada sobre galinhas para você:

Por que as galinhas são tão boas em jogos de cartas?

Porque elas sempre têm um "ovo" na mão!


# Componentes de um prompt

Os componentes de um prompt são os elementos estruturais de um comando para uma inteligência artificial generativa. Eles guiam o modelo e garantem que a resposta seja precisa, relevante e útil. Um prompt bem estruturado combina clareza, especificidade e contexto.

In [ ]:
# Componentes do prompt: Exemplo "Code Reviewer"

persona = "Você é um Engenheiro de Software Sênior especializado em Clean Code e arquitetura escalável.\n"

instruction = "Analise o trecho de código fornecido em busca de vulnerabilidades e oportunidades de refatoração.\n"

context = "Seu objetivo é garantir que o código siga os princípios SOLID e seja fácil de manter por outros desenvolvedores.\n"

data_format = "Crie uma lista de pontos de melhoria (bullets). Ao final, forneça uma versão refatorada do código que corrija os problemas apontados.\n"

audience = "A análise é voltada para um desenvolvedor júnior que está aprendendo boas práticas.\n"

tone = "O tom deve ser pedagógico, encorajador e técnico.\n"

# Código para análise
text = """
def process_data(data):
    r = []
    for item in data:
        if item['status'] == 'active':
            if item['value'] > 10:
                temp = item['value'] * 2
                r.append(temp)
    return r
"""

data = f"Código para analisar: {text}"

# O prompt completo
query = persona + instruction + context + data_format + audience + tone + data

output = pipe(query)
print(output[0]["generated_text"])



Vamos começar essa análise com um olhar atento ao código fornecido. Vamos explorar como ele segue ou não os princípios SOLID e quais possibilidades de melhoria existem. Vou explicar cada ponto de forma detalhada para que você possa entender melhor.

### Principais pontos de melhoria:

1. **Princípio da Responsabilidade Única (SRP)**:
   - O método `process_data` está fazendo duas coisas: filtrando itens com status 'active' e multiplicando valores maiores que 10. Isso pode tornar o código mais difícil de entender e manter. 

2. **Princípio da Aberto/Fechado (OP)**:
   - Não há nenhuma abertura para extensão, mas fechamento para modificação no código atual. Se precisarmos adicionar novas condições ou alterar a lógica, teremos que modificar o código.

3. **Princípio da Responsabilidade Conjunta (CRP)**:
   - O método `process_data` está responsável por várias tarefas. Podemos separar essas responsabilidades em métodos menores e mais focados.

4. **Princípio da Segregação de Interfaces (IS

# aprendizagem em contexto

Um LLM pode dar melhores resultados caso receba exemplos que podem utiliza para basear a sua resposta. Existem 3 categorias de tipos de aprendizagem:


*   Zero-shot

*   One-shot

*   Few-shot

In [ ]:
# Zero-Shot: Instrução direta sem exemplos prévios.
zero_shot_prompt = [
    {
        "role": "user",
        "content": "A palavra 'Zorgle' significa o ato de comer uma pizza muito rápido. Crie uma frase usando a palavra 'Zorgle':"
    }
]

# Gera o output

outputs = pipe(zero_shot_prompt)
print(outputs[0]["generated_text"])

"Eu adoro fazer um zorgle quando começo minha refeição de pizza!"


In [ ]:
# Técnica One-Shot para ensinar um conceito de sentimento inventado

one_shot_sentimento = [
    {
        "role": "user",
        "content": """'Zulbar' significa a sensação de felicidade misturada com saudade ao ver fotos antigas.
        Exemplo de frase usando Zulbar:"""
    },
    {
        "role": "assistant",
        "content": "Ao abrir o álbum de infância da minha mãe, senti um Zulbar profundo ao ver nossa casa antiga."
    },
    {
        "role": "user",
        "content": """'Klingson' significa o som que o gelo faz ao cair em um copo de vidro.
        Exemplo de frase usando Klingson:"""
    }
]

# Gera o output
outputs = pipe(one_shot_sentimento)
print(outputs[0]['generated_text'])

Quando o garoto jogou um cubo de gelo no copo de vidro, ele ouviu um Klingson que o fez rir por alguns minutos.


In [ ]:
# Few-Shot: Vários exemplos para fixar o padrão.

few_shot_prompt = [
    {
        "role": "user",
        "content": "Um 'Gigamuru' é um tipo de instrumento musical japonês. Frase:"
    },
    {
        "role": "assistant",
        "content": "Eu amo o som profundo do meu Gigamuru."
    },
    {
        "role": "user",
        "content": "'Screeg' significa golpear algo com uma espada. Frase:"
    },
    {
        "role": "assistant",
        "content": "O cavaleiro teve que screeg o dragão para defender o castelo."
    },
    {
        "role": "user",
        "content": "Um 'Blorp' é um líquido azul usado para limpar naves espaciais. Frase:"
    },
    {
        "role": "assistant",
        "content": "O capitão pediu mais Blorp para remover a poeira estelar das janelas."
    },
    {
        "role": "user",
        "content": "'Glip' significa pular sobre uma poça d'água. Frase:"
    }
]

# Gera o output

outputs = pipe(few_shot_prompt)
print(outputs[0]["generated_text"])

No campo de treinamento, o aluno decidiu glip antes de entrar na poça d'água para manter a calçada limpa.


# Chain prompting

Chain prompting é uma técnica que visa quebrar um problema em partes menores, nesse caso, dividir em uma sequencia de prompts para que o modelo possa ter uma melhor performance em casos mais complexos.

In [ ]:
# Criando um nome e slogam para um produto
product_prompt = [{"role": "user", "content": "Crie um nome e um slogan para um chatbot que utilize LLMs (Learning Lifecycle Management)."}]

outputs = pipe(product_prompt)
product_description = outputs[0]["generated_text"]
print(product_description)

Nome do Chatbot: LifespanAssistant

Slogan: "Compreendendo seu ciclo de vida, gerenciando seu sucesso." 

Este nome sugere que o chatbot é capaz de acompanhar e melhorar o desempenho ao longo do tempo, similar à maneira como um LLM (Learning Lifecycle Management) pode aprender e evoluir com base em interações e feedback. O slogan enfatiza a ideia de que o chatbot não apenas responde às perguntas ou problemas atuais, mas também ajuda a otimizar o processo ao longo do tempo.


In [ ]:
# Com base no nome e no slogan de um produto, elabore um discurso de vendas.
sales_prompt = [{"role": "user", "content": f"Elabore um discurso de vendas muito curto para o seguinte produto: '{product_description}'"}]

outputs = pipe(sales_prompt)
sales_pitch = outputs[0]["generated_text"]
print(sales_pitch)

Caros clientes,

Aproveitemos este momento para apresentar a vocês o LifespanAssistant, um chatbot revolucionário que não apenas responde às suas perguntas, mas também acompanha e melhora seu desempenho ao longo do tempo.

Com o LifespanAssistant, você pode contar com uma inteligência artificial que aprende e evolui com base nas interações diárias, assim como um LLM (Learning Lifecycle Management). Nós entendemos seu ciclo de vida e trabalhamos incansavelmente para maximizar seu sucesso.

"Compreendendo seu ciclo de vida, gerenciando seu sucesso." - Este é o nosso lema.

Junte-se a nós nesta jornada de otimização contínua. Com o LifespanAssistant, você está sempre um passo à frente.

Até a próxima!

[Alibaba Cloud]


# Chain-of-Thought:

Chain-of-Thought ou cadeia de raciocínio é uma tecnica que visa fazer o modelo "pensar" sobre o problema antes de dar uma resposta direta, para que assim a resposta seja mais precisa.

In [ ]:
# Respondendo sem uma cadeia de pensamento
No_cot_prompt = [
{"role": "user", "content": "A cantina tinha 23 maçãs. Se usaram 20 para fazer o almoço e compraram mais 6, quantas maçãs eles têm?"}
]

# Gerando a saída
outputs = pipe(No_cot_prompt)
print(outputs[0]["generated_text"])


Vamos resolver isso passo a passo:

1. A cantina começou com 23 maçãs.
2. Usaram 20 maçãs para fazer o almoço:
   23 - 20 = 3 maçãs restantes.
3. Depois disso, eles compraram mais 6 maçãs:
   3 + 6 = 9 maçãs.

Portanto, após usar algumas maçãs e comprar mais, a cantina tem agora 9 maçãs.


In [ ]:
# Respondendo com uma cadeia de pensamento
cot_prompt = [

{"role": "user", "content": """Roger tem 5 bolas de tênis. Ele compra mais 2 latas de bolas de tênis.
Cada lata contém 3 bolas de tênis. Quantas bolas de tênis ele tem agora?"""
},

{"role": "assistant", "content": "Roger começou com 5 bolas. Duas latas com 3 bolas de tênis cada totalizam 6 bolas de tênis. 5 + 6 = 11. A resposta é 11."},

{"role": "user", "content": "A cantina tinha 23 maçãs. Se usaram 20 para fazer o almoço e compraram mais 6, quantas maçãs eles têm??"}
]

# Gerando a saída
outputs = pipe(cot_prompt)
print(outputs[0]["generated_text"])

Antes de qualquer uso ou compra, havia 23 maçãs na cantina.

Depois de usar 20 maçãs para fazer o almoço, restaram:

23 - 20 = 3 maçãs

Depois que eles compraram mais 6 maçãs, o total passou a ser:

3 + 6 = 9 maçãs

Portanto, eles têm agora 9 maçãs.


Este exemplo mostra a utilização do zero-shot com a frase "Vamos pensar", este é um método comum para fazer com que o modelo realize a cadeia de ráciocínio:

In [ ]:
# Zero-shot chain-of-thought
zeroshot_cot_prompt = [{"role": "user", "content": """A cantina tinha 23 maçãs.
Se usaram 20 para fazer o almoço e compraram mais 6, quantas maçãs eles têm agora? Vamos pensar."""
}]

# Gerando a saída
outputs = pipe(zeroshot_cot_prompt)
print(outputs[0]["generated_text"])

Vamos resolver isso passo a passo:

1. A cantina começou com 23 maçãs.
2. Usaram 20 maçãs para fazer o almoço:
   23 - 20 = 3 maçãs restantes.
3. Depois disso, eles compraram mais 6 maçãs:
   3 + 6 = 9 maçãs.

Portanto, agora eles têm 9 maçãs.


# Tree-of-Thought

Árvore do Pensamento é um método que funciona da seguinte maneira: ao se deparar com um problema que exige múltiplas etapas de raciocínio, muitas vezes é útil dividi-lo em partes. modelo generativo é solicitado a explorar diferentes soluções para o problema em questão. Em seguida, ele vota na melhor solução e continua para a próxima etapa.

In [ ]:
# Zero-shot tree-of-thought
zeroshot_tot_prompt = [
{"role": "user", "content": '''
Imagine que três especialistas diferentes estão respondendo a esta pergunta.
Todos os especialistas anotarão uma etapa do seu raciocínio e a compartilharão com o grupo.
Em seguida, todos os especialistas passarão para a próxima etapa, e assim por diante. Se algum especialista perceber que está
errado em algum momento, ele se retira. A pergunta é: 'A cantina tinha 23 maçãs.

Se eles usaram 20 para fazer o almoço e compraram mais 6, quantas maçãs eles têm?'

Certifique-se de discutir os resultados.'''}
]

# Generando a saída
outputs = pipe(zeroshot_tot_prompt)
print(outputs[0]["generated_text"])

Claro, vamos proceder com esta tarefa. Vou começar com minha análise e depois compartilharei com os outros especialistas.

**Minha Análise:**
1. Inicialmente, a cantina tem 23 maçãs.
2. Usam 20 maçãs para fazer o almoço.
3. Então, a quantidade restante é 23 - 20 = 3 maçãs.
4. Depois, compram mais 6 maçãs.
5. Portanto, a quantidade total de maçãs agora é 3 + 6 = 9 maçãs.

**Especialista 1 (Matemático):**
1. Inicialmente, a cantina tem 23 maçãs.
2. Usam 20 maçãs para fazer o almoço.
   - 23 - 20 = 3 maçãs restantes.
3. Compram mais 6 maçãs.
   - 3 + 6 = 9 maçãs.
   
**Especialista 2 (Cientista de Dados):**
1. Inicialmente, a cantina tem 23 maçãs.
2. Usam 20 maçãs para fazer o almoço.
   - 23 - 20 = 3 maçãs restantes.
3. Compram mais 6 maçãs.
   - 3 + 6 = 9 maçãs.
   
**Especialista 3 (Físico):**
1. Inicialmente, a cantina tem 23 maçãs.
2. Usam 20 maçãs para fazer o almoço.
   - 23 - 20 = 3 maçãs restantes.
3. Compram mais 6 maçãs.
   - 3 + 6 = 9 maçãs.

Todos os especialistas chegaram à 

# Controlando o output

Geralmente, existem três maneiras de controlar a saída de um modelo generativo:

* Exemplos: Forneça vários exemplos da saída esperada.

* Gramática: Controle o processo de seleção de tokens.

* Ajuste fino (fine-tuning): Ajuste um modelo com dados que contenham a saída esperada.

In [ ]:
# Aprendizado Zero-shot: sem exemplos
zeroshot_prompt = [
{"role": "user", "content": "Crie um perfil de personagem para um jogo de RPG em formato JSON."}
]

# Gerando a saída
outputs = pipe(zeroshot_prompt)
print(outputs[0]["generated_text"])

Claro! Aqui está um exemplo de um perfil de personagem para um jogo de RPG em formato JSON:

```json
{
  "nome": "Evelyn",
  "classe": "Mago",
  "nivel": 10,
  "força": 12,
  "destreza": 15,
  "inteligência": 20,
  "vida": 60,
  "magia": 80,
  "experiência": 3500,
  "habilidades": [
    {
      "nome": "Feitiço de Fogo",
      "dano": 15,
      "uso": 1
    },
    {
      "nome": "Feitiço de Neve",
      "dano": 10,
      "uso": 1
    },
    {
      "nome": "Feitiço de Vento",
      "dano": 5,
      "uso": 1
    }
  ],
  "equipamentos": {
    "arma": "Espada Mágica",
    "armadura": "Escudo Mágico"
  },
  "objetos": [
    {
      "nome": "Pó de Estrela",
      "tipo": "Superalchimia",
      "efeito": "Aumenta a magia em 10%"
    },
    {
      "nome": "Mapa do Tesouro",
      "tipo": "Superalquimia",
      "efeito": "Mostra caminhos secretos e tesouros ocultos"
    }
  ]
}
```

Este perfil inclui informações como o nome do personagem, sua classe, nível, forças básicas (força, destreza,

In [ ]:
# Aprendizado One-shot: mostrando um exemplo de estrutura de output

one_shot_template = """Crie um breve perfil de personagem para um jogo de RPG. Certifique-se de usar apenas este formato:

{
"description": "UMA BREVE DESCRIÇÃO",
"nome": "O NOME DO PERSONAGEM",
"armadura": "UMA PEÇA DE ARMADURA",
"arma": "UMA OU MAIS ARMAS"
}
"""

one_shot_prompt = [
{"role": "user", "content": one_shot_template}
]

# Gerando a saída
outputs = pipe(one_shot_prompt)
print(outputs[0]["generated_text"])

{
"description": "Este personagem é um guerreiro valente e experiente, conhecido por sua força bruta e habilidade com espadas.",
"nome": "Thorvald",
"armadura": "Armadura de Ferro Avançada",
"arma": ["Espada de Trovão", "Escudo de Pedra"]
}
